<a href="https://colab.research.google.com/github/kuds/courtside-dynamics/blob/main/notebooks/sb3_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Courtside Dynamics: SB3 training

One notebook for the whole curriculum. Pick an environment (`BallBalance`, `BallBounce`, `WallBall`) and an algorithm (`SAC` or `PPO`) at the top, then run all cells.

Each environment's defaults (training budget, custom CSV rows, phase labels for the state-machine reward) live in `courtside_dynamics.recipes`, so adding an env is one entry in that registry -- this notebook needs no edits.

## 1. Install

In [ ]:
!pip install -q "courtside-dynamics[train,notebooks] @ git+https://github.com/kuds/courtside-dynamics"

## 2. Choose environment & algorithm

Set `ENV` and `ALGO` here -- everything below picks them up automatically.

* `USE_DRIVE = True` mounts Google Drive so checkpoints, eval npz, replay videos, and learning-curve plots survive a Colab runtime restart. Falls back to local logs if Drive isn't available.
* `QUICK_TEST = True` runs the whole pipeline (training, evaluation, video, plot) in a couple of minutes against a tiny budget. Use it to smoke-test a new runtime before committing to a real run.

### Suggested `n_envs` on Colab L4 (24 GB VRAM) + High-RAM (~51 GB)

These MuJoCo envs are CPU-cheap (tens of microseconds per step) and the MLP policy is small, so the GPU is never the bottleneck. The numbers below assume the default `DummyVecEnv` and the ~8 vCPUs Colab gives you.

| Algo | Suggested `n_envs` | Why |
|------|--------------------|-----|
| SAC  | **8**              | Off-policy. More envs fill the replay buffer faster, and the training helper sets `gradient_steps=-1` so updates scale 1:1 with steps collected regardless of `n_envs`. Past ~8 envs the CPU rollout cost dominates. |
| PPO  | **16**             | On-policy. The rollout buffer is `n_steps * n_envs`, so throughput scales close to linearly with `n_envs`. 16 fits Colab's CPU/RAM budget; bump to 32 if you switch to `SubprocVecEnv`. |

Override per run with `cfg.n_envs = ...` after `build_train_config(...)`, or pass `n_envs=...` as a kwarg to `build_train_config`.

In [ ]:
ENV = "WallBall"   # BallBalance | BallBounce | WallBall
ALGO = "SAC"         # SAC | PPO
USE_DRIVE = True
QUICK_TEST = False

# Master seed forwarded to SB3 and all helper envs; None for a
# nondeterministic run. Set it when comparing reward/env tweaks so
# run-to-run noise doesn't masquerade as a real difference.
SEED = None

# Training budget. None falls back to the recipe default (1M-2M steps);
# WallBall needs the longer 5M budget to get past contact-learning.
# Ignored when QUICK_TEST=True.
TOTAL_TIMESTEPS = 5_000_000


## 3. Mount Google Drive (optional) and pick a run directory

Each call to `resolve_run_dir(ENV, ALGO)` creates a fresh, timestamped directory so re-runs don't clobber prior artifacts. Layout:

```
<root>/<env>/<algo>/<YYYYMMDD_HHMMSS>/
  best_model.zip          final_model.zip
  evaluations.npz         monitor/*.monitor.csv
  tensorboard/            videos/
  checkpoints/            eval_info.csv
  vec_normalize.pkl       best_vec_normalize.pkl
  config.json             stage_summary.txt
  learning_curve.png      eval_info.png
  training_health.png     best_model.mp4
```

`checkpoints/` holds periodic full-state snapshots from `CheckpointCallback`; `eval_info.csv` is the long-format mirror of `InfoDictEvalCallback`'s TensorBoard scalars (timestep, metric, value). Two `VecNormalize` snapshots are written: `best_vec_normalize.pkl` captures the obs-normalization stats at the moment `best_model.zip` was saved, while `vec_normalize.pkl` holds the end-of-training stats. `record_best_model_video` prefers the former, so replay normalizes observations exactly the way the best model saw them.

Root is `MyDrive/Finding Theta/courtside-dynamics/training_runs/` when Drive is mounted, otherwise `./logs/`.


In [ ]:
from courtside_dynamics.notebook_utils import mount_drive, resolve_run_dir

if USE_DRIVE:
    mount_drive()

LOG_DIR = resolve_run_dir(ENV, ALGO, use_drive=USE_DRIVE)
print("Logging to:", LOG_DIR)

## 4. Configure Colab GPU

Sets up EGL so MuJoCo can render off-screen on the Colab GPU. No-op outside of Colab.

In [ ]:
from courtside_dynamics.colab_setup import setup_colab
setup_colab()

## 5. Build the training config

`build_train_config` looks up the recipe for `ENV`, fills in the per-env extras (e.g. custom CSV rows for Ball Bounce), and returns a `TrainConfig` ready for `train()`.

In [ ]:
from courtside_dynamics.recipes import RECIPES, build_train_config

print(f"Recipe: {ENV} -> {RECIPES[ENV].description}")

cfg = build_train_config(
    ENV,
    algo=ALGO,
    log_dir=LOG_DIR,
    total_timesteps=TOTAL_TIMESTEPS,
    quick_test=QUICK_TEST,
    seed=SEED,
)

# Long runs don't need a checkpoint/video every 250k steps; spread them
# out. Guarded so QUICK_TEST keeps the tight cadence from
# _QUICK_TEST_OVERRIDES and still exercises every callback.
if not QUICK_TEST:
    cfg.checkpoint_freq = 1_000_000
    cfg.video_freq = 1_000_000

print(
    f"algo={cfg.algo}  total_timesteps={cfg.total_timesteps:,}  "
    f"seed={cfg.seed}  eval_freq={cfg.eval_freq:,}  "
    f"checkpoint_freq={cfg.checkpoint_freq:,}  "
    f"video_freq={cfg.video_freq:,}"
)


## 5b. Live TensorBoard (optional)

Starts an inline TensorBoard tailing `LOG_DIR/tensorboard` so a multi-hour run can be checked mid-flight: eval reward under `eval/`, optimizer health under `train/`, per-eval info metrics under `eval_info/`. Scalars appear after SB3's first metric dump; use the refresh button. Safe to skip -- every scalar shown here is also mirrored to `progress.csv` / `eval_info.csv` and plotted statically in sections 7-8b.

In [ ]:
import os

from tensorboard import notebook as tb_notebook

# notebook.start shlex-parses its args, so the quotes keep Drive
# paths with spaces (MyDrive/Finding Theta/...) intact.
tb_notebook.start(f'--logdir "{os.path.join(LOG_DIR, "tensorboard")}"')


## 6. Train

`train(cfg)` builds vectorized train + eval envs, attaches `EvalCallback`, `VideoRecordCallback`, and `InfoDictEvalCallback`, and runs SB3's `model.learn`. The best policy seen during evaluation is saved to `LOG_DIR/best_model.zip`.

In [ ]:
from courtside_dynamics.training import train

model = train(cfg)

## 6b. Run report

`train()` writes `stage_summary.txt` at the end of every run -- final/best eval, wall-clock duration, throughput, device, and the final `train/*` health metrics -- including interrupted runs (`status: interrupted`). Printing it here attaches the numbers to this notebook session, and it's the first thing to paste when asking "why did this run underperform?".

In [ ]:
from courtside_dynamics.notebook_utils import print_stage_summary

print_stage_summary(LOG_DIR)


## 7. Learning curves

Per-episode training rewards (left) come from `LOG_DIR/monitor/*.monitor.csv`. Deterministic eval rewards (right, mean +/- std) come from `LOG_DIR/evaluations.npz`.

In [ ]:
import os
from courtside_dynamics.notebook_utils import plot_learning_curve

plot_learning_curve(
    LOG_DIR,
    save_path=os.path.join(LOG_DIR, "learning_curve.png"),
)

## 8. Eval-info curves

One panel per scalar `info` key tracked by `InfoDictEvalCallback` (rally count, paddle / wall touches, phase fractions, ...). `_mean`, `_final`, and `_max` variants are overlaid as separate lines per panel. The data comes from `LOG_DIR/eval_info.csv` (long-format mirror of the TensorBoard scalars, written every eval).

In [ ]:
from courtside_dynamics.notebook_utils import plot_eval_info

plot_eval_info(
    LOG_DIR,
    save_path=os.path.join(LOG_DIR, "eval_info.png"),
)

## 8b. Training-health curves

SB3's own optimizer diagnostics from `LOG_DIR/tensorboard/progress.csv`. For **SAC**: `ent_coef` (the entropy temperature — watch for it collapsing too fast or sticking high), plus `actor_loss` / `critic_loss` (a diverging critic is the classic failure). For **PPO**: `explained_variance` (below 0 means the value function is worse than predicting the mean), `approx_kl`, `clip_fraction`. These explain a stalled run that the reward curve alone won't.

In [ ]:
from courtside_dynamics.notebook_utils import plot_training_health

plot_training_health(
    LOG_DIR,
    save_path=os.path.join(LOG_DIR, "training_health.png"),
)

## 9. Replay the best model

Loads `best_model.zip` from `LOG_DIR`, rolls it out deterministically, encodes the frames as MP4, and embeds the clip in this notebook.

In [ ]:
from courtside_dynamics.notebook_utils import (
    record_best_model_video,
    display_video,
)

video_path = record_best_model_video(
    LOG_DIR,
    cfg.env_fn,
    algo=ALGO,
    video_length=750,
)
display_video(video_path)

## 9b. Artifact audit

Checks `LOG_DIR` against every artifact this notebook should have produced and prints the most likely cause for anything missing (video skipped because moviepy failed, no `best_model.zip` because eval never fired, ...). Run it **before** disconnecting: it's the last chance to re-run a failed cell while the runtime -- and everything not synced to Drive -- still exists.

In [ ]:
from courtside_dynamics.notebook_utils import check_run_artifacts

missing = check_run_artifacts(LOG_DIR)


## 10. Disconnect Colab runtime (optional)

Frees the GPU when you're done so the next run can grab a fresh runtime. Uncomment to enable. No-op outside of Colab.

In [ ]:
# Disconnect the Colab runtime when the run is over so the GPU is
# freed. Opt-in on purpose: run it only once the plots, replay video,
# and artifact audit above look right. No-op outside Colab.
# from courtside_dynamics.notebook_utils import disconnect_runtime
# disconnect_runtime()
